In [2]:
import pandas as pd

In [62]:
FILE_PATH = "path/to/your/file.csv"
df = pd.read_csv(FILE_PATH)

In [63]:
df_clear = df['org_type'].unique().dropna()

In [64]:
df_clear.size

1124

In [65]:
df_clear.tolist()

['Stadyum',
 'Kozmetik ve parfümeri mağazaları',
 'Locality',
 'Plastik ürün üreticileri',
 'Area',
 'District',
 'Restoran',
 'Dövmeciler',
 'Berberler',
 'Restaurant',
 'Diş sağlığı poliklinikleri',
 'Gümrükler',
 'Pasta, şekerleme ve tatlı',
 'Askerlik şubeleri',
 'Devlet kurumları ve bakanlıklar',
 'Güzellik salonu',
 'House',
 'Belediyeler, Devlet Daireleri',
 'Kafe',
 'Ortaokul',
 'Kültür ve eğlence parkları',
 'Finansal danışmanlık',
 'Alışveriş merkezleri',
 'Isıtma sistemlerinin kurulumu ve bakımı',
 'Otomobil servisi',
 'Müzeler ve sanat galerileri',
 'Telefon tamir servisi',
 'Fast food',
 'Lise',
 'Giyim mağazası',
 'Havaalanları',
 'Otopark ve tamirhaneler',
 'Taksi durağı',
 'Elektrik servisi',
 'Spa',
 'Hydro',
 'Street',
 'Süpermarket',
 'Elektronik eşya mağazaları',
 'Toptan gıda mağazaları',
 'Kuyumcular',
 'Tıp merkezleri ve klinikler',
 'İnşaat ve tasarım hizmetleri',
 'Çiçekçiler',
 'Otoparklar',
 'Other',
 'İskele',
 'Kütüphaneler',
 'Market',
 'Turistik yerler',


In [ ]:
import requests
from dotenv import load_dotenv
import os
load_dotenv()
API_KEY = os.getenv("YANDEX_TRANSLATE_API_KEY")
def translate_texts(texts):
    response = requests.post(
        "https://translate.api.cloud.yandex.net/translate/v2/translate",
        headers={
            "Authorization": f"Api-Key {API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "texts": texts,
            "targetLanguageCode": "en",
        },
    )
    data = response.json()
    translated = [item["text"] for item in data["translations"]]
    return translated

BATCH_SIZE = 100

translated = []
for i in range(0, len(df_clear), BATCH_SIZE):
    start = i
    end = min(i + BATCH_SIZE, len(df_clear))
    texts = df_clear[start:end].tolist()
    cur_translated = translate_texts(texts)
    translated.extend(cur_translated)

In [69]:
len(translated)

1124

In [70]:
from sentence_transformers import SentenceTransformer
MODEL_NAME = "BAAI/bge-large-en-v1.5"
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(translated).tolist()

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4637.42it/s]


In [71]:
org_type_tr = df_clear.tolist()
org_type_en = translated

translated_dict = dict()
for tr, en in zip(org_type_tr, org_type_en):
    translated_dict[tr] = en

In [72]:
translated_dict

{'Stadyum': 'Stadium',
 'Kozmetik ve parfümeri mağazaları': 'Cosmetics and perfumery stores',
 'Locality': 'Locality',
 'Plastik ürün üreticileri': 'Plastic product manufacturers',
 'Area': 'Area',
 'District': 'District',
 'Restoran': 'Restaurant',
 'Dövmeciler': 'Tattooists',
 'Berberler': 'Barbers',
 'Restaurant': 'The Restaurant',
 'Diş sağlığı poliklinikleri': 'Dental health outpatient clinics',
 'Gümrükler': 'Customs',
 'Pasta, şekerleme ve tatlı': 'Cake, candy and dessert',
 'Askerlik şubeleri': 'Military service branches',
 'Devlet kurumları ve bakanlıklar': 'Government agencies and ministries',
 'Güzellik salonu': 'Beauty salon',
 'House': 'House',
 'Belediyeler, Devlet Daireleri': 'Municipalities, Government Offices',
 'Kafe': 'Cafe',
 'Ortaokul': 'Secondary school',
 'Kültür ve eğlence parkları': 'Culture and amusement parks',
 'Finansal danışmanlık': 'Financial consultancy',
 'Alışveriş merkezleri': 'Shopping malls',
 'Isıtma sistemlerinin kurulumu ve bakımı': 'Installation

In [73]:
from scipy.spatial import distance
list_of_categories = [
    "Agriculture",
    "Automotive, motorcycles and vehicles",
    "Beauty, personal care and wellness",
    "Construction, renovation and interior",
    "Culture, arts and entertainment",
    "Education and training",
    "Finance and professional services",
    "Food and beverages",
    "Government and municipal services",
    "Healthcare and medical services",
    "IT, telecommunications and electronics",
    "Landmarks, addresses and geography",
    "Manufacturing, industry and equipment",
    "Nature, parks and outdoor places",
    "Other and unspecified",
    "Real estate and business properties",
    "Religion and community organizations",
    "Retail and trade",
    "Sports and active recreation",
    "Tourism, lodging and travel",
    "Transportation and logistics",
    "Utilities, security and maintenance services",
]

model = SentenceTransformer(MODEL_NAME)
cat_emb = model.encode(list_of_categories).tolist()
org_type_cat = {}
for i in range(len(embeddings)):
    eb = embeddings[i]
    best_ind = 0
    best_dist = 2
    for j in range(len(list_of_categories)):
        current_dist = distance.cosine(eb, cat_emb[j])
        if current_dist < best_dist:
            best_dist = current_dist
            best_ind = j
    org_type_cat[translated[i]] = list_of_categories[best_ind]
print(org_type_cat)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4201.98it/s]


{'Stadium': 'Sports and active recreation', 'Cosmetics and perfumery stores': 'Beauty, personal care and wellness', 'Locality': 'Landmarks, addresses and geography', 'Plastic product manufacturers': 'Manufacturing, industry and equipment', 'Area': 'Nature, parks and outdoor places', 'District': 'Government and municipal services', 'Restaurant': 'Food and beverages', 'Tattooists': 'Beauty, personal care and wellness', 'Barbers': 'Retail and trade', 'The Restaurant': 'Food and beverages', 'Dental health outpatient clinics': 'Healthcare and medical services', 'Customs': 'Other and unspecified', 'Cake, candy and dessert': 'Food and beverages', 'Military service branches': 'Government and municipal services', 'Government agencies and ministries': 'Government and municipal services', 'Beauty salon': 'Beauty, personal care and wellness', 'House': 'Agriculture', 'Municipalities, Government Offices': 'Government and municipal services', 'Cafe': 'Food and beverages', 'Secondary school': 'Educati

In [74]:
import numpy as np
translated_dict[np.nan] = np.nan
org_type_cat[np.nan] = 'Other and unspecified'

In [75]:
df_cats = [org_type_cat[translated_dict[tr]] for tr in df['org_type']]

In [76]:
df['category'] = df_cats

In [77]:
df.head()

,latitude,longitude,model_response_timestamp,query,region_name,org_found,org_name,org_type,org_rating,org_lat,org_lon,geometry,index_right,name,number,category
0,37.874904,32.492129,1771686554,Konumumdan en yakın rotayı çıkar,Konya,True,Konya Büyükşehir Belediye Stadyumu,Stadyum,4.9,37.946209,32.488097,POINT (32.492129 37.874904),52,Konya,42,Sports and active recreation
1,41.011219,28.978176,1772400724,Takı Dünyası Nerdedir?,Istanbul,True,Takı Dünyası,Kozmetik ve parfümeri mağazaları,NaN,41.009805,28.694718,POINT (28.978176 41.011219),39,İstanbul,34,"Beauty, personal care and wellness"
2,41.011219,28.978176,1771065812,Şuan bulunduğum konumdan bursa görükle ye gide...,Istanbul,False,NaN,NaN,NaN,NaN,NaN,POINT (28.978176 41.011219),39,İstanbul,34,Other and unspecified
3,37.062688,37.379510,1771742482,Milano da gezilecek yerler bana söyler misin?,Gaziantep,False,NaN,NaN,NaN,NaN,NaN,POINT (37.37951 37.062688),32,Gaziantep,27,Other and unspecified
4,37.181190,33.222247,1771836177,karaman adana arası kaç km ve tır ne yakar,Karaman,True,Karaman,Locality,NaN,37.181193,33.222241,POINT (33.222247 37.18119),43,Karaman,70,"Landmarks, addresses and geography"


In [78]:
df.to_csv("df_with_cat.csv")

In [79]:
org_type_cat

{'Stadium': 'Sports and active recreation',
 'Cosmetics and perfumery stores': 'Beauty, personal care and wellness',
 'Locality': 'Landmarks, addresses and geography',
 'Plastic product manufacturers': 'Manufacturing, industry and equipment',
 'Area': 'Nature, parks and outdoor places',
 'District': 'Government and municipal services',
 'Restaurant': 'Food and beverages',
 'Tattooists': 'Beauty, personal care and wellness',
 'Barbers': 'Retail and trade',
 'The Restaurant': 'Food and beverages',
 'Dental health outpatient clinics': 'Healthcare and medical services',
 'Customs': 'Other and unspecified',
 'Cake, candy and dessert': 'Food and beverages',
 'Military service branches': 'Government and municipal services',
 'Government agencies and ministries': 'Government and municipal services',
 'Beauty salon': 'Beauty, personal care and wellness',
 'House': 'Agriculture',
 'Municipalities, Government Offices': 'Government and municipal services',
 'Cafe': 'Food and beverages',
 'Secondar